### Notebook to Search and Filter Networking Nature Data for Lime Keywords

In [3]:
import pandas as pd
import re
from pathlib import Path

In [4]:
import urllib.parse

# --- Keywords ---
WATER_KEYWORDS = [
    "shore", "ocean", "sea", "river", "lake", "pond", "stream", "creek",
    "wetland", "reservoir", "estuary", "gulf", "bay",
    "body of water", "bodies of water"
]

# Per-keyword patterns (word boundaries; multi-word phrases work naturally)
kw_patterns = {
    kw: re.compile(r"\b" + re.escape(kw) + r"\b", flags=re.IGNORECASE)
    for kw in WATER_KEYWORDS
}

# Combined filter pattern (non-capturing group avoids UserWarning)
combined_pattern = re.compile(
    r"\b(?:" + "|".join(re.escape(kw) for kw in WATER_KEYWORDS) + r")\b",
    flags=re.IGNORECASE
)

# Extracts institution code and volume ID from source_file name
# e.g. "blackwood_v24_1828_inu-30000080770807-1746739609.txt" -> ("inu", "30000080770807")
source_file_pattern = re.compile(r'_([a-z]+)-(\d+)-\d+\.txt$')

def make_hathitrust_link(source_file, seq, kw_counts):
    m = source_file_pattern.search(str(source_file))
    if not m:
        return ""
    institution, volume_id = m.group(1), m.group(2)
    # Use the keyword with the highest occurrence count; fall back to first matched
    best_kw = max(kw_counts, key=lambda k: kw_counts[k])
    q1 = urllib.parse.quote(best_kw)
    return f"https://babel.hathitrust.org/cgi/pt?id={institution}.{volume_id}&seq={seq}&q1={q1}"

# --- Paths ---
input_dir = Path("input_data")
output_dir = Path("output_data")
output_dir.mkdir(exist_ok=True)

csv_files = sorted(input_dir.glob("*.csv"))
print(f"Found {len(csv_files)} input files: {[f.name for f in csv_files]}")

# --- Filter, annotate, and save ---
for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    text_col = df["text"].astype(str)

    # Keep only rows that contain at least one keyword
    mask = text_col.str.contains(combined_pattern)
    filtered = df[mask].reset_index(drop=True)
    text_filtered = filtered["text"].astype(str)

    # One column per keyword: count of occurrences in the text (0 if absent)
    kw_col_names = []
    for kw, pat in kw_patterns.items():
        col = "kw_" + kw.replace(" ", "_")
        filtered[col] = text_filtered.apply(lambda t, p=pat: len(p.findall(t)))
        kw_col_names.append(col)

    # Summary: number of distinct keywords matched and a readable list
    filtered["keyword_count"] = filtered[kw_col_names].gt(0).sum(axis=1)
    filtered["keywords_found"] = filtered[kw_col_names].apply(
        lambda row: ", ".join(kw for kw, col in zip(WATER_KEYWORDS, kw_col_names) if row[col] > 0),
        axis=1
    )

    # HathiTrust document link: seq = half_page_number * 2, q1 = most present keyword
    filtered["document_link"] = filtered.apply(
        lambda row: make_hathitrust_link(
            row["source_file"],
            int(row["half_page_number"]) * 2,
            {kw: row[col] for kw, col in zip(WATER_KEYWORDS, kw_col_names)}
        ),
        axis=1
    )

    out_path = output_dir / f"{csv_path.stem}_water_filtered.csv"
    filtered.to_csv(out_path, index=False)
    print(f"{csv_path.name}: {len(df)} rows -> {len(filtered)} kept ({len(filtered)/len(df):.1%}) -> {out_path.name}")

Found 4 input files: ['blackwood_sci_topics_lime.csv', 'cook_sci_topics_lime.csv', 'isis_sci_topics_lime.csv', 'pmg_sci_topics_lime.csv']
blackwood_sci_topics_lime.csv: 322 rows -> 67 kept (20.8%) -> blackwood_sci_topics_lime_water_filtered.csv
cook_sci_topics_lime.csv: 423 rows -> 91 kept (21.5%) -> cook_sci_topics_lime_water_filtered.csv
isis_sci_topics_lime.csv: 402 rows -> 29 kept (7.2%) -> isis_sci_topics_lime_water_filtered.csv
pmg_sci_topics_lime.csv: 303 rows -> 27 kept (8.9%) -> pmg_sci_topics_lime_water_filtered.csv
